# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL for machine-readable reproducibility and access.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The metadata attribute gives object access (not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Dataset ID: {dataset.metadata.id}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets and the fields/columns contained within them. Each is referenced by its `@id` for robust, schema-driven exploration.

Let's enumerate all record sets, then review their fields and columns.

In [ ]:
# List all available record sets and display their IDs and field IDs

recordset_ids = []
if hasattr(dataset.metadata, 'record_sets'):
    for rs in dataset.metadata.record_sets:
        print(f"RecordSet: {getattr(rs, 'id', None)}  Name: {getattr(rs, 'name', None)}")
        recordset_ids.append(getattr(rs, 'id', None))
        # List fields and columns for each record set
        print("  Fields:")
        for f in getattr(rs, 'fields', []):
            print(f"    Field: {getattr(f, 'id', None)} | Name: {getattr(f, 'name', None)} | Type: {getattr(f, 'data_type', None)}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for c in getattr(rs, 'columns', []):
                print(f"    Column: {getattr(c, 'id', None)} | Name: {getattr(c, 'name', None)}")
else:
    print("No record sets defined in schema; please check dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the above overview.


In [ ]:
# For this dataset, let's list all record set IDs
record_sets = []
if hasattr(dataset.metadata, 'record_sets'):
    record_sets = [getattr(rs, 'id', None) for rs in dataset.metadata.record_sets]

# Prepare dataframes for each record set (by @id)
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records for RecordSet: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Display columns of the first record set for illustration
if len(dataframes):
    first_rs = list(dataframes.keys())[0]
    print('Columns available:', dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print('No data frames loaded (no accessible record sets or data)')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.


In [ ]:
# Example EDA: select a numeric field, filter, normalize, and group

# Use the first available DataFrame and try to select a numeric field for analysis
import numpy as np
if len(dataframes):
    df = list(dataframes.values())[0]
    record_set_id = list(dataframes.keys())[0]
    # Find the first numeric column for demonstration
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if numeric_field is not None:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a column containing 'type', 'class', or 'group', else skip
        group_field = None
        for col in df.columns:
            if any(sub in col.lower() for sub in ['type', 'class', 'group']):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print('\nNo clear grouping field found in columns')
    else:
        print('No numeric field found for EDA.')
else:
    print('No data loaded into DataFrames for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
# Plot histogram of the selected numeric field if available
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field available for histogram.')

## 6. Conclusion
In this notebook, the FAIR^2 dataset was loaded and its structure explored via the Croissant schema. We examined available record sets, loaded tabular data, and performed basic EDA including filtering, normalization, and visualization.

- For detailed analysis, investigate the domain meaning of each record set and column (refer to the field `@id`s)
- The Croissant schema-driven approach ensures reproducibility and semantic clarity
- Further work can include domain-specific aggregation and advanced analysis leveraging the dataset's metadata provenance
